# Gender Bias in Occupation Prediction: A BiasBios Analysis

## Project Overview
This project uses the BiasBios dataset to train a logistic regression classifier that predicts occupation from professional biography text. Beyond standard evaluation, the project includes an ethics analysis: the top-weighted TF-IDF features per occupation class are extracted from the trained model, and their correlation with ground-truth gender labels is computed. This surfaces the gendered vocabulary driving predictions, demonstrating how gender bias can persist in a model even when gender is not an explicit input feature.

## Imports & Setup

In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.core.display import HTML

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score

from datasets import load_dataset
from pathlib import Path

import os
import random
from enum import StrEnum

# Allow to show all columns
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

# Set seaborn theme easy to read for colorblind readers
plt.style.use("ggplot")
sns.set_theme(style="whitegrid")


In [ ]:
# Globals
DATA_PATH = Path("biasbios.parquet")
NOTEBOOK_SEED = 42

In [16]:
# Ensure Reproducibility
np.random.seed(NOTEBOOK_SEED)
random.seed(NOTEBOOK_SEED)

`BiasInBiosColumns` are used to reduce typos in accessing columns in the pandas DateFrame:

In [17]:
# Description of Dataset columns

class BiasInBiosColumns(StrEnum):
    """Column names for the Bias in Bios dataset.

    Source: https://huggingface.co/datasets/LabHC/bias_in_biostrain
    License: MIT
    """

    # Core columns
    HARD_TEXT = "hard_text"
    PROFESSION = "profession"
    GENDER = "gender"
    SPLIT = "split"
    
class DataSplit(StrEnum):
    """Valid Values for column BiasInBiosColumns.SPLIT"""
    TRAIN = "train"
    DEV = "dev"
    TEST = "test"

VALID_SPLITS = {s.value for s in DataSplit}

## Load Data 

Automatically download dataset if it does not exist. In any case `biasbios_df` will contain a pandas DataFrame with Bias Bios Dataset: 

In [20]:
def download_bias_in_bios(data_path: Path):
    """Download bias_in_bios dataset and save to CSV.
    
    Args:
        data_path: Path object where CSV will be written.
        
    Raises:
        AssertionError: If load_dataset().to_pandas() returns non-DataFrame.
    """
     
    dfs : list[pd.DataFrame] = []
    for split in VALID_SPLITS:
        df = load_dataset("LabHC/bias_in_bios", split=split).to_pandas()

        # can return an iterator but it shouldn't assert corrrectness 
        assert isinstance(df, pd.DataFrame), "Incorrect Dataset"
        df[BiasInBiosColumns.SPLIT] = split
        dfs.append(df)
    
    full_df = pd.concat(dfs, ignore_index=True, axis=0) 
    assert isinstance(full_df, pd.DataFrame), "Incorrect Dataset"
    # create CSV 
    data_path.parent.mkdir(parents=True, exist_ok=True)
    full_df.to_parquet(data_path) 

def load_and_cache_data(data_path: Path) -> pd.DataFrame:

    """Load bias_in_bios dataset, downloading and caching if necessary.

    Args:
        data_path: Path where cached CSV is/will be stored.

    Returns:
        DataFrame with bias_in_bios data and split column.
    """
    if not data_path.exists():
        download_bias_in_bios(data_path)

    return pd.read_parquet(data_path)

In [21]:
biasbios_df = load_and_cache_data(DATA_PATH)

In [13]:
biasbios_df.head()

,Unnamed: 0,hard_text,profession,gender,split
0,0,"He specializes in development economics, house...",21,0,test
1,1,He started out as a DJ and music producer in t...,5,0,test
2,2,"She is averse to all things scary or sad, so s...",4,1,test
3,3,"Prior to joining USC, she was a mobile news ed...",21,1,test
4,4,"Previously, she served as an assistant profess...",21,1,test
